# Notebook 13: Custom Loss Functions

## 📚 Historical Context

**Timeline**: 2012-2023 - Evolution from simple cross-entropy to specialized losses

**The Journey of Loss Functions**:
- **2012**: AlexNet - Standard cross-entropy for everything
  - Problem: Treats all classes equally
  - Problem: Overconfident predictions
  - Works OK when data is balanced

- **2017**: Focal Loss (RetinaNet paper)
  - Motivation: Object detection with extreme imbalance (99.9% background)
  - Solution: Focus on hard examples, down-weight easy ones
  - Impact: Enabled single-stage detectors to match two-stage

- **2016-2019**: Label Smoothing (Inception v2 → widespread adoption)
  - Problem: One-hot labels → overconfident predictions → poor calibration
  - Solution: Smooth labels (0.1 to other classes instead of 0)
  - Used in: Modern ImageNet training, many transformers

- **2015**: Knowledge Distillation (Hinton et al.)
  - Problem: Large models too slow for deployment
  - Solution: Train small model to mimic large model's soft predictions
  - Impact: DistilBERT, TinyBERT, MobileBERT

- **2020+**: Multi-task Learning
  - Single model, multiple objectives
  - Challenge: Balancing loss scales
  - Used in: T5, GPT-4, modern LLMs

**Why This Matters**:
Cross-entropy is a **starting point**, not the **final answer**. Real-world problems often need custom losses:
- Medical diagnosis: Class imbalance (1% disease)
- Object detection: 99% background pixels
- Fine-tuning: Knowledge distillation from larger models
- Multi-task: Sentiment + topic classification together

## 🎯 What You'll Learn

1. **Cross-Entropy Variants** - Weighted, class-balanced
2. **Focal Loss** - For extreme imbalance
3. **Label Smoothing** - Prevent overconfidence
4. **Knowledge Distillation** - Learn from teacher models
5. **Multi-Task Losses** - Combine multiple objectives
6. **Debugging Loss Issues** - NaN, explosion, convergence

## 💡 Why This Matters

Understanding loss functions is CRITICAL because:
- **Wrong loss → Model learns wrong thing**
- **Imbalanced data + standard CE → Model ignores minority class**
- **No label smoothing → Overconfident, poorly calibrated**
- **Can't debug training without understanding loss**

**Real-world example**: You're building a fraud detection system. Fraud is 0.1% of transactions. Standard cross-entropy? Model learns to predict "not fraud" for everything → 99.9% accuracy but catches ZERO fraud. Need focal loss or weighted CE.

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Understanding Cross-Entropy

### The Standard Loss Function:

**Mathematical Definition**:
```
CE = -Σ y_i * log(ŷ_i)

For classification:
CE = -log(ŷ_true_class)
```

Where:
- `y_i`: True label (one-hot encoded)
- `ŷ_i`: Predicted probability
- `ŷ_true_class`: Predicted probability for the correct class

### Intuition:

**What does it optimize for?**
```
True label: Cat
Model predicts: P(cat)=0.9, P(dog)=0.1
Loss = -log(0.9) = 0.10  ← Small loss, good!

Model predicts: P(cat)=0.1, P(dog)=0.9
Loss = -log(0.1) = 2.30  ← Large loss, bad!

Model predicts: P(cat)=0.01, P(dog)=0.99
Loss = -log(0.01) = 4.61  ← Very large loss, very bad!
```

### PyTorch Implementation:

**Two ways**:
```python
# 1. nn.CrossEntropyLoss() - includes softmax
#    Input: raw logits [batch, num_classes]
#    Target: class indices [batch]
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)

# 2. Manual - when you need more control
log_probs = F.log_softmax(logits, dim=-1)
loss = F.nll_loss(log_probs, labels)
```

### When Standard CE Fails:

**1. Class imbalance**: 99 samples class A, 1 sample class B
- Model learns to ignore class B
- 99% accuracy by always predicting A

**2. Overconfidence**: Predicts 0.9999 for everything
- Low training loss
- Poor calibration (confidence ≠ accuracy)
- Brittle to distribution shift

**3. Hard negatives**: Some examples are inherently difficult
- Easy examples dominate gradient
- Hard examples ignored

In [ ]:
def visualize_cross_entropy():
    """
    Visualize how cross-entropy loss changes with predicted probability.
    """
    # Predicted probabilities from 0.01 to 0.99
    probs = np.linspace(0.01, 0.99, 100)
    
    # Cross-entropy loss = -log(p)
    ce_loss = -np.log(probs)
    
    plt.figure(figsize=(10, 5))
    
    plt.plot(probs, ce_loss, linewidth=2, color='blue')
    plt.xlabel('Predicted Probability for True Class', fontsize=12)
    plt.ylabel('Cross-Entropy Loss', fontsize=12)
    plt.title('Cross-Entropy Loss vs Predicted Probability', fontsize=14)
    plt.grid(True, alpha=0.3)
    
    # Annotate key points
    key_points = [0.1, 0.5, 0.9]
    for p in key_points:
        loss = -np.log(p)
        plt.scatter([p], [loss], s=100, c='red', zorder=5)
        plt.annotate(
            f'p={p:.1f}\nloss={loss:.2f}',
            xy=(p, loss),
            xytext=(p + 0.1, loss + 0.5),
            fontsize=10,
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
            arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0')
        )
    
    plt.tight_layout()
    plt.show()

print("="*60)
print("CROSS-ENTROPY LOSS BEHAVIOR")
print("="*60)
visualize_cross_entropy()

print("\nKey Observations:")
print("1. Loss → ∞ as predicted probability → 0")
print("2. Loss → 0 as predicted probability → 1")
print("3. Non-linear: Small improvement at high confidence costs a lot")
print("4. Symmetric: Treats all mistakes equally")
print("="*60)

## Part 2: Weighted Cross-Entropy

### The Problem:

**Imbalanced dataset**:
```
Class A: 900 samples
Class B: 100 samples
Total:   1000 samples
```

**What happens with standard CE?**
```
Model: Always predict class A
Accuracy: 90%  ← Looks good!
Recall for B: 0%  ← Actually useless!
```

### Solution: Weight Rare Classes More:

**Class weights**:
```
weight_i = total_samples / (num_classes * samples_in_class_i)

Or inverse frequency:
weight_i = 1 / frequency_i

Or balanced:
weight_i = total_samples / (num_classes * samples_in_class_i)
```

**Example**:
```
Class A: 900 samples → weight = 1000 / (2 * 900) = 0.56
Class B: 100 samples → weight = 1000 / (2 * 100) = 5.0
```

**Effect**:
- Misclassifying B costs 5.0/0.56 ≈ 9x more than misclassifying A
- Model learns to care about minority class

### PyTorch Implementation:

```python
# Compute class weights
class_counts = [900, 100]
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum()  # Normalize

# Create weighted loss
criterion = nn.CrossEntropyLoss(weight=class_weights)
```

### When to Use:
- Moderate imbalance (10:1 to 100:1)
- All classes are important
- Simple to implement

### When NOT to Use:
- Extreme imbalance (>1000:1) → Use focal loss
- Noisy labels → Weights amplify noise
- Already using balanced sampling

In [ ]:
# Create imbalanced dataset
class ImbalancedDataset(Dataset):
    """Dataset with severe class imbalance."""
    def __init__(self, num_samples, imbalance_ratio=0.1, num_features=20):
        """
        Args:
            num_samples: Total number of samples
            imbalance_ratio: Ratio of minority class (e.g., 0.1 = 10%)
            num_features: Feature dimension
        """
        self.num_samples = num_samples
        self.num_features = num_features
        
        # Determine number of samples per class
        num_minority = int(num_samples * imbalance_ratio)
        num_majority = num_samples - num_minority
        
        # Generate data
        # Majority class (0): centered at origin
        majority_data = torch.randn(num_majority, num_features) * 0.5
        majority_labels = torch.zeros(num_majority, dtype=torch.long)
        
        # Minority class (1): shifted to make it learnable
        minority_data = torch.randn(num_minority, num_features) * 0.5 + 2.0
        minority_labels = torch.ones(num_minority, dtype=torch.long)
        
        # Combine
        self.data = torch.cat([majority_data, minority_data], dim=0)
        self.labels = torch.cat([majority_labels, minority_labels], dim=0)
        
        # Shuffle
        indices = torch.randperm(num_samples)
        self.data = self.data[indices]
        self.labels = self.labels[indices]
        
        print(f"Dataset created:")
        print(f"  Total samples: {num_samples}")
        print(f"  Class 0 (majority): {num_majority} ({num_majority/num_samples*100:.1f}%)")
        print(f"  Class 1 (minority): {num_minority} ({num_minority/num_samples*100:.1f}%)")
        print(f"  Imbalance ratio: {num_majority/num_minority:.1f}:1")
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    
    def get_class_counts(self):
        """Return counts for each class."""
        unique, counts = torch.unique(self.labels, return_counts=True)
        return counts.tolist()

# Simple classifier
class BinaryClassifier(nn.Module):
    """Simple binary classifier."""
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 2)  # Binary classification
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

def train_and_evaluate(model, train_loader, val_loader, criterion, num_epochs=10, name=""):
    """Train model and return metrics."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    history = {'train_loss': [], 'val_acc': [], 'val_recall_minority': []}
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        
        # Metrics
        accuracy = (all_preds == all_labels).mean()
        
        # Recall for minority class (class 1)
        minority_mask = all_labels == 1
        if minority_mask.sum() > 0:
            recall_minority = (all_preds[minority_mask] == 1).mean()
        else:
            recall_minority = 0.0
        
        history['train_loss'].append(train_loss)
        history['val_acc'].append(accuracy)
        history['val_recall_minority'].append(recall_minority)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, "
                  f"Acc={accuracy:.4f}, Minority Recall={recall_minority:.4f}")
    
    return history, all_preds, all_labels

# Create datasets
print("\n" + "="*60)
print("COMPARING STANDARD vs WEIGHTED CROSS-ENTROPY")
print("="*60)

train_dataset = ImbalancedDataset(num_samples=1000, imbalance_ratio=0.1)
val_dataset = ImbalancedDataset(num_samples=200, imbalance_ratio=0.1)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# 1. Standard Cross-Entropy
print("\n1. Training with STANDARD Cross-Entropy:")
print("-" * 60)
model_standard = BinaryClassifier(input_dim=20).to(device)
criterion_standard = nn.CrossEntropyLoss()

history_standard, preds_standard, labels_standard = train_and_evaluate(
    model_standard, train_loader, val_loader, criterion_standard, num_epochs=20
)

# 2. Weighted Cross-Entropy
print("\n2. Training with WEIGHTED Cross-Entropy:")
print("-" * 60)

# Compute class weights (inverse frequency)
class_counts = train_dataset.get_class_counts()
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights / class_weights.sum() * len(class_weights)  # Normalize
print(f"Class weights: {class_weights.tolist()}")

model_weighted = BinaryClassifier(input_dim=20).to(device)
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights.to(device))

history_weighted, preds_weighted, labels_weighted = train_and_evaluate(
    model_weighted, train_loader, val_loader, criterion_weighted, num_epochs=20
)

# Compare results
print("\n" + "="*60)
print("COMPARISON")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Minority class recall
axes[0].plot(history_standard['val_recall_minority'], label='Standard CE', marker='o')
axes[0].plot(history_weighted['val_recall_minority'], label='Weighted CE', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Minority Class Recall')
axes[0].set_title('Minority Class Recall Over Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Confusion matrices
cm_standard = confusion_matrix(labels_standard, preds_standard)
cm_weighted = confusion_matrix(labels_weighted, preds_weighted)

axes[1].text(0.1, 0.8, 'Standard CE:', fontsize=12, weight='bold', transform=axes[1].transAxes)
axes[1].text(0.1, 0.7, f'Minority Recall: {history_standard["val_recall_minority"][-1]:.3f}', 
             fontsize=11, transform=axes[1].transAxes)
axes[1].text(0.1, 0.6, f'Overall Acc: {history_standard["val_acc"][-1]:.3f}', 
             fontsize=11, transform=axes[1].transAxes)

axes[1].text(0.1, 0.4, 'Weighted CE:', fontsize=12, weight='bold', transform=axes[1].transAxes)
axes[1].text(0.1, 0.3, f'Minority Recall: {history_weighted["val_recall_minority"][-1]:.3f}', 
             fontsize=11, transform=axes[1].transAxes)
axes[1].text(0.1, 0.2, f'Overall Acc: {history_weighted["val_acc"][-1]:.3f}', 
             fontsize=11, transform=axes[1].transAxes)

axes[1].axis('off')
axes[1].set_title('Final Metrics')

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("Weighted CE significantly improves minority class recall!")
print("Standard CE often ignores minority class entirely.")
print("="*60)

## Part 3: Focal Loss

### The Problem with Weighted CE:

**Extreme imbalance** (e.g., object detection):
```
Background: 1,000,000 pixels
Object:           1,000 pixels
Ratio: 1000:1
```

Even with class weights:
- Easy examples (obvious background) still dominate
- Model spends effort on what it already knows
- Hard examples (object boundaries) get insufficient attention

### Focal Loss Innovation (Lin et al., 2017):

**Idea**: Down-weight easy examples, focus on hard ones.

**Mathematical Formula**:
```
FL = -α * (1 - p_t)^γ * log(p_t)

Where:
- p_t: Predicted probability for true class
- α: Class weight (optional, typically 0.25)
- γ: Focusing parameter (typically 2.0)

Standard CE:
CE = -log(p_t)
```

### How It Works:

**The (1 - p_t)^γ term is the key**:

```
Easy example (p_t = 0.9, model is confident):
(1 - 0.9)^2 = 0.01
→ Loss reduced by 100x!

Hard example (p_t = 0.5, model is uncertain):
(1 - 0.5)^2 = 0.25
→ Loss reduced by 4x only

Very hard (p_t = 0.1, model is wrong):
(1 - 0.1)^2 = 0.81
→ Loss barely reduced
```

**Effect**: Model focuses gradient on hard examples.

### Hyperparameters:

**γ (gamma)** - Focusing parameter:
- γ = 0: Reduces to standard CE
- γ = 2: Recommended default
- γ = 5: Very aggressive focusing

**α (alpha)** - Class weight:
- α = 0.25 for positive class (minority)
- α = 0.75 for negative class (majority)
- Or set to None (no class weighting)

### When to Use:
- Extreme imbalance (>100:1)
- Many easy examples dominating
- Object detection, semantic segmentation
- When weighted CE isn't enough

### Papers Using Focal Loss:
- RetinaNet (object detection)
- EfficientDet
- Many modern detection architectures

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss implementation.
    
    Reference: "Focal Loss for Dense Object Detection" (Lin et al., 2017)
    https://arxiv.org/abs/1708.02002
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        Args:
            alpha: Class weights [num_classes]. If None, no weighting.
            gamma: Focusing parameter. Higher = more focus on hard examples.
            reduction: 'mean', 'sum', or 'none'
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        """
        Args:
            inputs: Logits [batch_size, num_classes]
            targets: Class indices [batch_size]
        """
        # Get probabilities
        probs = F.softmax(inputs, dim=-1)
        
        # Get probability for true class
        batch_size = inputs.size(0)
        p_t = probs[range(batch_size), targets]  # [batch_size]
        
        # Compute focal term: (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma
        
        # Compute CE loss: -log(p_t)
        ce_loss = -torch.log(p_t + 1e-8)  # Add epsilon for numerical stability
        
        # Focal loss: focal_weight * ce_loss
        focal_loss = focal_weight * ce_loss
        
        # Apply class weights if provided
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        # Reduction
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

def visualize_focal_loss():
    """
    Compare Focal Loss with different gamma values vs standard CE.
    """
    probs = np.linspace(0.01, 0.99, 100)
    
    # Standard CE
    ce_loss = -np.log(probs)
    
    # Focal loss with different gammas
    gammas = [0, 1, 2, 5]
    
    plt.figure(figsize=(12, 5))
    
    for gamma in gammas:
        focal_loss = -((1 - probs) ** gamma) * np.log(probs)
        label = 'Standard CE' if gamma == 0 else f'Focal (γ={gamma})'
        plt.plot(probs, focal_loss, label=label, linewidth=2)
    
    plt.xlabel('Predicted Probability for True Class', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Focal Loss: Effect of Focusing Parameter γ', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Key Observations:")
    print("1. γ=0: Identical to standard CE")
    print("2. γ=2: Default, good balance")
    print("3. γ=5: Very aggressive, focuses almost only on hard examples")
    print("4. At high confidence (p>0.9), loss is dramatically reduced")
    print("5. At low confidence (p<0.5), loss is similar to CE")

print("\n" + "="*60)
print("FOCAL LOSS VISUALIZATION")
print("="*60)
visualize_focal_loss()

# Test on imbalanced dataset
print("\n" + "="*60)
print("TESTING FOCAL LOSS ON IMBALANCED DATA")
print("="*60)

# Create even more imbalanced dataset (5% minority)
train_dataset_extreme = ImbalancedDataset(num_samples=1000, imbalance_ratio=0.05)
val_dataset_extreme = ImbalancedDataset(num_samples=200, imbalance_ratio=0.05)

train_loader_extreme = DataLoader(train_dataset_extreme, batch_size=32, shuffle=True)
val_loader_extreme = DataLoader(val_dataset_extreme, batch_size=32, shuffle=False)

print("\nTraining with Focal Loss (γ=2):")
print("-" * 60)

model_focal = BinaryClassifier(input_dim=20).to(device)
criterion_focal = FocalLoss(gamma=2.0)

history_focal, preds_focal, labels_focal = train_and_evaluate(
    model_focal, train_loader_extreme, val_loader_extreme, 
    criterion_focal, num_epochs=20
)

print("\n" + "="*60)
print(f"Final Minority Class Recall: {history_focal['val_recall_minority'][-1]:.4f}")
print(f"Final Overall Accuracy: {history_focal['val_acc'][-1]:.4f}")
print("\nFocal Loss excels at extreme imbalance!")
print("="*60)

## Part 4: Label Smoothing

### The Problem: Overconfidence

**Standard one-hot labels**:
```
True label: Cat
One-hot:    [0, 1, 0]  (dog, cat, bird)

Model learns to predict:
[0.001, 0.998, 0.001]  ← Very confident!
```

**Why is this bad?**
1. **Poor calibration**: Confidence ≠ Accuracy
   - Says 99.8% confident, but might be wrong
   - Critical for medical, autonomous driving

2. **Overfitting**: Encourages extreme predictions
   - Pushes logits to infinity
   - Poor generalization

3. **Distribution shift**: Brittle to new data
   - Train on ImageNet, test on ImageNet-C
   - Confidence doesn't match reality

### Solution: Label Smoothing

**Smooth the hard labels**:
```
Standard one-hot:
[0, 1, 0]  ← 100% certain it's cat

Label smoothing (ε=0.1):
[0.05, 0.9, 0.05]  ← 90% cat, 5% each other class
```

**Mathematical Formula**:
```
y_smooth = (1 - ε) * y + ε / num_classes

Where:
- y: One-hot label
- ε: Smoothing parameter (typically 0.1)
- num_classes: Number of classes

Example with ε=0.1, num_classes=3:
True class: 1 - 0.1 + 0.1/3 = 0.933
Other classes: 0 + 0.1/3 = 0.033
```

### Benefits:

1. **Better calibration**: Predictions match accuracy
2. **Regularization**: Prevents overfitting
3. **Robustness**: Better generalization
4. **Teacher for distillation**: Soft labels transfer knowledge

### Typical Values:
- ε = 0.1: Standard for ImageNet
- ε = 0.2: More aggressive smoothing
- ε = 0.0: No smoothing (standard CE)

### Used In:
- Inception v2 (2016) - First popularized
- Modern ImageNet training
- BERT, GPT-2, T5 (some variants)
- Many competition-winning models

### PyTorch Implementation:

**Option 1**: Manual
```python
def label_smoothing_loss(logits, targets, smoothing=0.1):
    confidence = 1.0 - smoothing
    log_probs = F.log_softmax(logits, dim=-1)
    
    # True class
    nll_loss = -log_probs.gather(dim=-1, index=targets.unsqueeze(1))
    nll_loss = nll_loss.squeeze(1)
    
    # Smooth distribution
    smooth_loss = -log_probs.mean(dim=-1)
    
    # Combine
    loss = confidence * nll_loss + smoothing * smooth_loss
    return loss.mean()
```

**Option 2**: Custom loss class (cleaner)

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy loss with label smoothing.
    
    Reference: "Rethinking the Inception Architecture for Computer Vision"
    (Szegedy et al., 2016)
    """
    def __init__(self, smoothing=0.1, reduction='mean'):
        """
        Args:
            smoothing: Label smoothing parameter (ε)
            reduction: 'mean', 'sum', or 'none'
        """
        super().__init__()
        self.smoothing = smoothing
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        """
        Args:
            inputs: Logits [batch_size, num_classes]
            targets: Class indices [batch_size]
        """
        num_classes = inputs.size(-1)
        log_probs = F.log_softmax(inputs, dim=-1)
        
        # Loss for true class
        # gather() extracts log_prob for the true class
        nll_loss = -log_probs.gather(dim=-1, index=targets.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)  # [batch_size]
        
        # Smooth loss (average over all classes)
        smooth_loss = -log_probs.mean(dim=-1)  # [batch_size]
        
        # Combine with label smoothing
        # loss = (1-ε) * nll_loss + ε * smooth_loss
        loss = (1.0 - self.smoothing) * nll_loss + self.smoothing * smooth_loss
        
        # Reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

def compare_confidence_calibration():
    """
    Show how label smoothing affects model confidence.
    """
    # Create balanced dataset for cleaner comparison
    class BalancedDataset(Dataset):
        def __init__(self, num_samples, num_features=20, num_classes=3):
            self.data = torch.randn(num_samples, num_features)
            self.labels = torch.randint(0, num_classes, (num_samples,))
        
        def __len__(self):
            return len(self.data)
        
        def __getitem__(self, idx):
            return self.data[idx], self.labels[idx]
    
    class MultiClassifier(nn.Module):
        def __init__(self, input_dim, num_classes):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 64)
            self.fc2 = nn.Linear(64, num_classes)
        
        def forward(self, x):
            x = F.relu(self.fc1(x))
            return self.fc2(x)
    
    # Train two models
    train_data = BalancedDataset(1000)
    val_data = BalancedDataset(200)
    
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32)
    
    # Model 1: Standard CE
    print("\nTraining with Standard Cross-Entropy:")
    model_standard = MultiClassifier(20, 3).to(device)
    criterion_standard = nn.CrossEntropyLoss()
    optimizer_standard = torch.optim.AdamW(model_standard.parameters(), lr=1e-3)
    
    for epoch in range(10):
        model_standard.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_standard.zero_grad()
            outputs = model_standard(inputs)
            loss = criterion_standard(outputs, labels)
            loss.backward()
            optimizer_standard.step()
    
    # Model 2: Label Smoothing
    print("Training with Label Smoothing (ε=0.1):")
    model_smooth = MultiClassifier(20, 3).to(device)
    criterion_smooth = LabelSmoothingCrossEntropy(smoothing=0.1)
    optimizer_smooth = torch.optim.AdamW(model_smooth.parameters(), lr=1e-3)
    
    for epoch in range(10):
        model_smooth.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_smooth.zero_grad()
            outputs = model_smooth(inputs)
            loss = criterion_smooth(outputs, labels)
            loss.backward()
            optimizer_smooth.step()
    
    # Compare confidence distributions
    model_standard.eval()
    model_smooth.eval()
    
    confidences_standard = []
    confidences_smooth = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            
            # Standard model
            outputs_std = model_standard(inputs)
            probs_std = F.softmax(outputs_std, dim=-1)
            max_conf_std = probs_std.max(dim=-1)[0]
            confidences_standard.extend(max_conf_std.cpu().numpy())
            
            # Smoothed model
            outputs_smooth = model_smooth(inputs)
            probs_smooth = F.softmax(outputs_smooth, dim=-1)
            max_conf_smooth = probs_smooth.max(dim=-1)[0]
            confidences_smooth.extend(max_conf_smooth.cpu().numpy())
    
    # Plot confidence distributions
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(confidences_standard, bins=20, alpha=0.7, label='Standard CE', color='blue')
    plt.axvline(np.mean(confidences_standard), color='blue', linestyle='--', 
                label=f'Mean: {np.mean(confidences_standard):.3f}')
    plt.xlabel('Maximum Confidence')
    plt.ylabel('Frequency')
    plt.title('Standard Cross-Entropy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.hist(confidences_smooth, bins=20, alpha=0.7, label='Label Smoothing', color='green')
    plt.axvline(np.mean(confidences_smooth), color='green', linestyle='--',
                label=f'Mean: {np.mean(confidences_smooth):.3f}')
    plt.xlabel('Maximum Confidence')
    plt.ylabel('Frequency')
    plt.title('Label Smoothing (ε=0.1)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nObservations:")
    print(f"Standard CE - Average confidence: {np.mean(confidences_standard):.3f}")
    print(f"Label Smooth - Average confidence: {np.mean(confidences_smooth):.3f}")
    print("\nLabel smoothing produces lower, more calibrated confidence.")
    print("This is GOOD - predictions better reflect actual accuracy!")

print("\n" + "="*60)
print("LABEL SMOOTHING: CONFIDENCE CALIBRATION")
print("="*60)
compare_confidence_calibration()
print("="*60)

## Part 5: Knowledge Distillation Loss

### The Goal:

**Problem**: Large model (teacher) works great but is too slow/big for deployment.

**Solution**: Train small model (student) to mimic teacher's behavior.

### Key Insight:

**Teacher's soft predictions contain more information than hard labels**:

```
Image: Husky dog

Hard label:
[0, 0, 1, 0, 0]  (cat, wolf, husky, fox, tiger)
→ Only says "it's a husky"

Teacher's soft predictions:
[0.02, 0.15, 0.70, 0.08, 0.05]
→ Says "mainly husky, but has wolf-like features,
   a bit fox-like, definitely not cat or tiger"
```

**The soft predictions encode relationships**: huskies are more similar to wolves than cats.

### Knowledge Distillation Loss:

**Two components**:
```
Total Loss = α * KD_loss + (1-α) * CE_loss

Where:
KD_loss = KL_divergence(student_probs || teacher_probs)
CE_loss = CrossEntropy(student_logits, true_labels)
α = distillation weight (typically 0.7-0.9)
```

**Temperature scaling** (makes soft targets softer):
```python
soft_targets = softmax(teacher_logits / T)
soft_student = softmax(student_logits / T)

T = 1: Standard softmax (sharp)
T = 3: Softer (more information about similarities)
T = 10: Very soft (mostly uniform)
```

### Why Temperature?

**Example with T=1 (standard)**:
```
Logits: [2.0, 1.0, 0.5]
Softmax: [0.66, 0.24, 0.10]
```

**With T=3 (softer)**:
```
Scaled logits: [0.67, 0.33, 0.17]
Softmax: [0.45, 0.32, 0.23]
```

Higher T → More information in non-maximum classes.

### Applications:
- **DistilBERT**: 6 layers vs BERT's 12, 97% performance, 2x faster
- **TinyBERT**: 4 layers, 96% performance, 10x faster
- **MobileBERT**: Optimized for mobile devices
- **Model compression**: Deploy on edge devices

### Typical Hyperparameters:
- Temperature: 3-10 (higher for more classes)
- α (distillation weight): 0.7-0.9
- Often train student for more epochs than teacher

In [ ]:
class DistillationLoss(nn.Module):
    """
    Knowledge Distillation Loss.
    
    Reference: "Distilling the Knowledge in a Neural Network" (Hinton et al., 2015)
    """
    def __init__(self, temperature=3.0, alpha=0.7):
        """
        Args:
            temperature: Temperature for softening probabilities
            alpha: Weight for distillation loss (1-alpha for CE loss)
        """
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, student_logits, teacher_logits, labels):
        """
        Args:
            student_logits: Student model outputs [batch, num_classes]
            teacher_logits: Teacher model outputs [batch, num_classes]
            labels: True labels [batch]
        """
        # Soften logits with temperature
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=-1)
        soft_student = F.log_softmax(student_logits / self.temperature, dim=-1)
        
        # KL divergence loss (distillation loss)
        # Multiply by T^2 to scale gradients correctly
        distillation_loss = F.kl_div(
            soft_student, 
            soft_teacher, 
            reduction='batchmean'
        ) * (self.temperature ** 2)
        
        # Standard cross-entropy loss with true labels
        student_loss = self.ce_loss(student_logits, labels)
        
        # Combined loss
        total_loss = (
            self.alpha * distillation_loss + 
            (1 - self.alpha) * student_loss
        )
        
        return total_loss

def demonstrate_distillation():
    """
    Demonstrate knowledge distillation with teacher-student setup.
    """
    print("\n" + "="*60)
    print("KNOWLEDGE DISTILLATION DEMONSTRATION")
    print("="*60)
    
    # Dataset
    class SimpleDataset(Dataset):
        def __init__(self, num_samples, num_features=20, num_classes=5):
            self.data = torch.randn(num_samples, num_features)
            self.labels = torch.randint(0, num_classes, (num_samples,))
        
        def __len__(self):
            return len(self.data)
        
        def __getitem__(self, idx):
            return self.data[idx], self.labels[idx]
    
    # Models
    class TeacherModel(nn.Module):
        """Large teacher model."""
        def __init__(self, input_dim, num_classes):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 256)
            self.fc2 = nn.Linear(256, 256)
            self.fc3 = nn.Linear(256, 128)
            self.fc4 = nn.Linear(128, num_classes)
            self.dropout = nn.Dropout(0.2)
        
        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = F.relu(self.fc2(x))
            x = self.dropout(x)
            x = F.relu(self.fc3(x))
            x = self.fc4(x)
            return x
    
    class StudentModel(nn.Module):
        """Small student model (3x smaller)."""
        def __init__(self, input_dim, num_classes):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 64)
            self.fc2 = nn.Linear(64, num_classes)
        
        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return x
    
    # Prepare data
    train_data = SimpleDataset(1000)
    val_data = SimpleDataset(200)
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32)
    
    # Step 1: Train teacher model
    print("\n1. Training Teacher Model (large):")
    print("-" * 60)
    teacher = TeacherModel(20, 5).to(device)
    teacher_params = sum(p.numel() for p in teacher.parameters())
    print(f"Teacher parameters: {teacher_params:,}")
    
    optimizer_teacher = torch.optim.AdamW(teacher.parameters(), lr=1e-3)
    criterion_teacher = nn.CrossEntropyLoss()
    
    for epoch in range(15):
        teacher.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_teacher.zero_grad()
            outputs = teacher(inputs)
            loss = criterion_teacher(outputs, labels)
            loss.backward()
            optimizer_teacher.step()
    
    # Evaluate teacher
    teacher.eval()
    correct_teacher = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = teacher(inputs)
            _, preds = torch.max(outputs, 1)
            correct_teacher += (preds == labels).sum().item()
            total += labels.size(0)
    
    teacher_acc = correct_teacher / total
    print(f"Teacher Accuracy: {teacher_acc:.4f}")
    
    # Step 2: Train student WITHOUT distillation (baseline)
    print("\n2. Training Student WITHOUT Distillation (baseline):")
    print("-" * 60)
    student_baseline = StudentModel(20, 5).to(device)
    student_params = sum(p.numel() for p in student_baseline.parameters())
    print(f"Student parameters: {student_params:,} ({student_params/teacher_params:.1%} of teacher)")
    
    optimizer_baseline = torch.optim.AdamW(student_baseline.parameters(), lr=1e-3)
    criterion_baseline = nn.CrossEntropyLoss()
    
    for epoch in range(15):
        student_baseline.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer_baseline.zero_grad()
            outputs = student_baseline(inputs)
            loss = criterion_baseline(outputs, labels)
            loss.backward()
            optimizer_baseline.step()
    
    student_baseline.eval()
    correct_baseline = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = student_baseline(inputs)
            _, preds = torch.max(outputs, 1)
            correct_baseline += (preds == labels).sum().item()
    
    baseline_acc = correct_baseline / total
    print(f"Student (baseline) Accuracy: {baseline_acc:.4f}")
    
    # Step 3: Train student WITH distillation
    print("\n3. Training Student WITH Distillation:")
    print("-" * 60)
    student_distilled = StudentModel(20, 5).to(device)
    optimizer_distilled = torch.optim.AdamW(student_distilled.parameters(), lr=1e-3)
    criterion_distillation = DistillationLoss(temperature=3.0, alpha=0.7)
    
    teacher.eval()  # Keep teacher in eval mode
    for epoch in range(15):
        student_distilled.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Get teacher outputs (no gradient)
            with torch.no_grad():
                teacher_outputs = teacher(inputs)
            
            # Train student
            optimizer_distilled.zero_grad()
            student_outputs = student_distilled(inputs)
            loss = criterion_distillation(student_outputs, teacher_outputs, labels)
            loss.backward()
            optimizer_distilled.step()
    
    student_distilled.eval()
    correct_distilled = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = student_distilled(inputs)
            _, preds = torch.max(outputs, 1)
            correct_distilled += (preds == labels).sum().item()
    
    distilled_acc = correct_distilled / total
    print(f"Student (distilled) Accuracy: {distilled_acc:.4f}")
    
    # Summary
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    print(f"Teacher (large):           {teacher_acc:.4f} ({teacher_params:,} params)")
    print(f"Student (baseline):        {baseline_acc:.4f} ({student_params:,} params)")
    print(f"Student (distilled):       {distilled_acc:.4f} ({student_params:,} params)")
    print(f"\nImprovement from distillation: {(distilled_acc - baseline_acc):.4f}")
    print(f"Model size reduction: {student_params/teacher_params:.1%}")
    print("\nKey Insight: Distillation helps student learn from teacher's")
    print("soft predictions, achieving better performance with fewer parameters!")
    print("="*60)

demonstrate_distillation()

## 📊 Summary

### Loss Functions Covered:

**1. Standard Cross-Entropy**
- Formula: `CE = -log(p_t)`
- Use: Balanced datasets, baseline
- Limitations: Ignores imbalance, overconfident

**2. Weighted Cross-Entropy**
- Formula: `WCE = -w_c * log(p_t)`
- Use: Moderate imbalance (10:1 to 100:1)
- Implementation: `nn.CrossEntropyLoss(weight=class_weights)`

**3. Focal Loss**
- Formula: `FL = -α * (1-p_t)^γ * log(p_t)`
- Use: Extreme imbalance (>100:1), many easy examples
- Hyperparameters: γ=2 (focusing), α=0.25 (class weight)

**4. Label Smoothing**
- Formula: `y_smooth = (1-ε)*y + ε/K`
- Use: Prevent overconfidence, improve calibration
- Typical value: ε=0.1

**5. Knowledge Distillation**
- Formula: `Loss = α*KL(student||teacher) + (1-α)*CE`
- Use: Model compression, transfer learning
- Hyperparameters: T=3 (temperature), α=0.7

### Decision Tree: Which Loss to Use?

```
Start
  |
  ├─ Balanced dataset?
  │   ├─ Yes → Label Smoothing CE (ε=0.1)
  │   └─ No → Continue
  |
  ├─ Imbalance ratio?
  │   ├─ <10:1 → Label Smoothing + Class Weights
  │   ├─ 10:1 to 100:1 → Weighted CE
  │   └─ >100:1 → Focal Loss (γ=2)
  |
  ├─ Model compression?
  │   └─ Yes → Knowledge Distillation
  |
  └─ Multiple tasks?
      └─ Yes → Multi-task loss (next section)
```

### Common Pitfalls & Solutions:

**Problem: Loss becomes NaN**
- Cause: log(0) or division by zero
- Solution: Add epsilon `log(p + 1e-8)`
- Solution: Gradient clipping
- Solution: Lower learning rate

**Problem: Loss explodes**
- Cause: Learning rate too high
- Solution: LR warmup
- Solution: Gradient clipping (max_norm=1.0)
- Solution: Check input normalization

**Problem: Loss plateaus immediately**
- Cause: Model predicting majority class only
- Solution: Use weighted CE or focal loss
- Solution: Check class balance
- Solution: Verify model actually learning (check gradients)

**Problem: Good training loss, bad validation**
- Cause: Overfitting
- Solution: Add label smoothing
- Solution: Increase dropout
- Solution: Reduce model size or add regularization

### Production Checklist:

✓ **Analyze dataset first**
  - Check class distribution
  - Identify imbalance ratio
  - Compute class weights

✓ **Start simple**
  - Baseline: Standard CE
  - Add label smoothing (almost always helps)
  - If imbalanced, try weighted CE
  - If still bad, try focal loss

✓ **Monitor metrics beyond loss**
  - Per-class accuracy/recall
  - Confusion matrix
  - Confidence calibration

✓ **Debug systematically**
  - Log loss value every N steps
  - Check for NaN/Inf
  - Verify gradient flow
  - Test on small batch first

### Next Notebooks:

**Notebook 14: Handling Imbalanced Classification**
- Binary imbalanced example (95/5 split)
- Comparing all techniques
- Threshold tuning
- Evaluation metrics for imbalanced data

**Notebook 15: Mixed Precision Training**
- FP16/BF16 training
- Automatic Mixed Precision (AMP)
- Numerical stability with custom losses
- Memory and speed benchmarks

---

**Key Takeaway**: There's no one-size-fits-all loss function. Understanding your data and task is critical for choosing the right loss. Start simple, measure carefully, and iterate.